In [2]:
library(tidyverse)
library(dbplyr)
library(bigrquery)
library(lubridate)

bq_auth()

project_id = "yhcr-prd-bradfor-bia-core"

# create connection to database
con <- DBI::dbConnect(bigrquery::bigquery(), 
                      project = project_id)

print(paste0("Connected to : ", project_id))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘dbplyr’


The following objects are masked from ‘package:dplyr’:

    ident, sql




[1] "Connected to : yhcr-prd-bradfor-bia-core"


# Read in person cohort csv

In [176]:
person <- read.csv('data/person_cohorts.csv', header = TRUE)

In [177]:
head(person)

,person_id,birth_date,gender,CombinedEthnicity,cohort
,<chr>,<chr>,<chr>,<chr>,<chr>
1,D27CE553B8DA52E66BCBA43420F6F53DAEBF9FE2864361849339028B6B002695,2000-09-15,F,South Asian,2000/01
2,9F33F6D292805042AF92186788B71C7302C7E976F6C148C34FDE62B157C03928,2000-09-15,F,South Asian,2000/01
3,C38AD0B73560547A95C529E42A9CF6E1E06282163BB419DC2C1451422CD20CE0,2000-09-15,F,South Asian,2000/01
4,03A81FF3A58E65546A42780919E74680BF016F6D3E8FE1FA64FFD9B9E436B795,2000-09-15,F,NA,2000/01
5,8C9979BE485657CDE5AC92361D6E454DC61C1396BCC38F77933217649F802EFF,2000-09-15,F,White British,2000/01
6,00ED2E8CBC0AE48B370DBB70FB9440A00BD65113392DE33A5E14380D5C306075,2000-09-15,F,White British,2000/01


In [178]:
person |>
nrow()

[1] 22101

In [179]:
person_ids <- read.csv('data/person_ids.csv', header = TRUE)

In [180]:
head(person_ids)

,person_id,NCCIS_ACADYR
,<chr>,<chr>
1,0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018
2,00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2017/2018
3,0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018/2019
4,0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2018/2019
5,00149D26484ED3BE2DEF154B8DB53A36B1A9FF0839C447A2CE19B0750225BA66,2017/2018
6,0014DF7EADF7C22A0FD2F6862B6F06778B199F1F2D61242125DA8EC539E6275D,2017/2018


In [181]:
person_ids |>
nrow()

[1] 17361

In [182]:
# Check if all person_ids in person_id are in person
all(person_ids$person_id %in% person$person_id)

[1] TRUE

In [183]:
# filter person to only matching IDs
person_filtered <- person |>
    filter(person_id %in% person_ids$person_id)

In [184]:
person_filtered |>
nrow()

[1] 17361

In [185]:
all(person_ids$person_id %in% person_filtered$person_id)

[1] TRUE

# Siblings / home indicators

In [186]:
school_census_data = "CB_2489.cb_StudentAdditionalInfoFromCensus"

In [187]:
home_table <- tbl(con, school_census_data) |>
    select(person_id, SourceTable, DistCurrSch, DistNearSch, ModeOfTravel, Boarder, Language, IDACIScore_Home, LSOA11_Home, AdoptedFromCare, InCare, NSiblings_SGA, BirthOrder_SGA, GroupID_SGA) |>
    filter(!is.na(person_id))

Auto-refreshing stale OAuth token.



In [188]:
# SGA means based on shared address (vs. on address and surname)
# BirthOrder_SGA 1 is the oldest

In [189]:
home_table

# Source:   SQL [?? x 14]
# Database: BigQueryConnection
   person_id   SourceTable DistCurrSch DistNearSch ModeOfTravel Boarder Language
   <chr>       <chr>             <dbl>       <dbl> <chr>        <chr>   <chr>   
 1 1599DBDF7E… Spring_Cen…        0.19        0.19 NA           N       HGR     
 2 767B260381… Spring_Cen…        0.93        0.25 NA           N       PRSD    
 3 4351EA3E9D… Spring_Cen…        0.52        0.28 NA           N       CZE     
 4 806A603473… Spring_Cen…        4.78       NA    NA           N       ARA     
 5 A1E1773668… Spring_Cen…        2.61        0.2  NA           N       PRS     
 6 94D9517B50… Spring_Cen…        0.15        0.12 NA           N       SOM     
 7 DF5B98D405… Spring_Cen…        0.13        0.13 NA           N       ALB     
 8 E6763D54CB… Spring_Cen…        0.18        0.18 NA           N       ENG     
 9 CAA9275D57… Spring_Cen…        3.1        NA    NA           N       ENG     
10 1C02B5DDD6… Spring_Cen…        0.53        0.23 N

In [190]:
home_table_df <- home_table |>
    collect()

In [191]:
home_table_df |>
    nrow()

[1] 6656974

In [192]:
home_table_df |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
302369


In [193]:
home_table_filtered <- home_table_df |>
    filter(person_id %in% person_ids$person_id)

In [194]:
home_table_filtered |>
    nrow()

[1] 599106

In [195]:
home_table_filtered |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17224


In [196]:
home_table_filtered |>
    filter(person_id == '0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD')

person_id,SourceTable,DistCurrSch,DistNearSch,ModeOfTravel,Boarder,Language,IDACIScore_Home,LSOA11_Home,AdoptedFromCare,InCare,NSiblings_SGA,BirthOrder_SGA,GroupID_SGA
<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Autumn_Census_2014,NA,NA,NA,N,ENG,0.32444,E01010635,NA,NA,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Spring_Census_2012,0.15,0.15,NA,N,ENG,0.32444,NA,NA,NA,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Spring_Census_2016,0.66,0.66,NA,N,ENG,NA,E01010635,N,NA,2,1,733638
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Autumn_Census_2012,NA,NA,NA,N,ENG,NA,NA,NA,NA,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,PLASC_2005,0.15,NA,NA,N,NA,0.37623,NA,NA,0,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Summer_Census_2013,NA,NA,NA,N,ENG,0.32444,E01010635,NA,NA,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Summer_Census_2009,NA,NA,NA,N,ENG,0.35747,NA,NA,NA,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Summer_Census_2016,NA,NA,NA,N,ENG,NA,E01010635,N,NA,NA,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Spring_Census_2013,0.66,0.66,NA,N,ENG,0.32444,E01010635,NA,NA,2,1,535577


In [197]:
# extract year from source table
home_table_filtered$Year <- substr(home_table_filtered$SourceTable, nchar(home_table_filtered$SourceTable) - 3, nchar(home_table_filtered$SourceTable))

In [198]:
head(home_table_filtered)

person_id,SourceTable,DistCurrSch,DistNearSch,ModeOfTravel,Boarder,Language,IDACIScore_Home,LSOA11_Home,AdoptedFromCare,InCare,NSiblings_SGA,BirthOrder_SGA,GroupID_SGA,Year
<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
90BB0A71C61559E8739BB61BB06DF5219E61B68C1473EF196EE2E636FA7EBD66,Spring_Census_2020,0.57,NA,NA,N,ENG,NA,E01010805,NA,NA,NA,NA,NA,2020
63D51806BF39CD8BA75B167A225A6265CACB64C2BCC450D1CC6A8D1B7762C51A,Summer_Census_2007,NA,NA,NA,N,ENG,0.62848,NA,NA,NA,NA,NA,NA,2007
7E85C178F3746E87FD206106D7DE8A60C6A3673D85F289AFA7260A41DBFD52C8,Summer_Census_2007,NA,NA,NA,N,ENG,0.58800,NA,NA,NA,NA,NA,NA,2007
E625DCFFBF76BE04A4AF39A037ECC571BC1E9DB022EF1866EA592F0F453E897E,Summer_Census_2007,NA,NA,NA,N,OTB,0.38284,NA,NA,NA,NA,NA,NA,2007
8FD16984BBF636D0A880EBDE8C0752AF282E2FEDE469927D33EA1C8A87CB4102,Summer_Census_2007,NA,NA,NA,N,URD,0.35253,NA,NA,NA,NA,NA,NA,2007
52F485644A0C4A92B167EA8FAD081A9F07FD525D52E6DA83F5FBA5386AE9B528,Autumn_Census_2008,NA,NA,NA,N,ARA,0.35479,NA,NA,NA,NA,NA,NA,2008


In [199]:
# drop source table col and then merge / remove duplicate rows
home_table_filtered <- home_table_filtered |>
    select(-SourceTable)

In [200]:
home_table_filtered <- home_table_filtered |>
    distinct()

In [201]:
home_table_filtered |>
    nrow()

[1] 507335

In [202]:
home_table_filtered |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17224


#### drop cols

In [203]:
home_table_filtered |> distinct(ModeOfTravel) |> pull(ModeOfTravel)

[1] NA    "CRS" "CAR" "WLK" "DSB" "CYC" "TXI" "BNK" "PSB" "LUL" "TRN" "OTH"
[13] "MTL" "BDR"

In [204]:
home_table_filtered |> distinct(Boarder) |> pull(Boarder)

[1] "N" "6" "B" NA  "7"

In [205]:
home_table_filtered |> distinct(AdoptedFromCare) |> pull(AdoptedFromCare)

[1] NA  "N" "A" "R" "G"

In [206]:
home_table_filtered |> distinct(InCare) |> pull(InCare)

[1] NA    "0"   "1"   "810" "380" "384" "888" "382" "381" "316" "883" "856"
[13] "391" "359" "383" "352" "815" "850" "841" "343"

In [207]:
home_table_filtered <- home_table_filtered |>
    select(-c(ModeOfTravel,Boarder,AdoptedFromCare,InCare,NSiblings_SGA,BirthOrder_SGA,GroupID_SGA))

In [208]:
# add in cohort identifier so that can drop data from primary school ranges
home_table_merge <- home_table_filtered |>
  left_join(person_filtered |> select(person_id, cohort), by = "person_id")

In [209]:
home_table_merge |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17224


In [210]:
head(home_table_merge)

person_id,DistCurrSch,DistNearSch,Language,IDACIScore_Home,LSOA11_Home,Year,cohort
<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>
90BB0A71C61559E8739BB61BB06DF5219E61B68C1473EF196EE2E636FA7EBD66,0.57,NA,ENG,NA,E01010805,2020,2000/01
63D51806BF39CD8BA75B167A225A6265CACB64C2BCC450D1CC6A8D1B7762C51A,NA,NA,ENG,0.62848,NA,2007,2000/01
7E85C178F3746E87FD206106D7DE8A60C6A3673D85F289AFA7260A41DBFD52C8,NA,NA,ENG,0.58800,NA,2007,2000/01
E625DCFFBF76BE04A4AF39A037ECC571BC1E9DB022EF1866EA592F0F453E897E,NA,NA,OTB,0.38284,NA,2007,2000/01
8FD16984BBF636D0A880EBDE8C0752AF282E2FEDE469927D33EA1C8A87CB4102,NA,NA,URD,0.35253,NA,2007,2000/01
52F485644A0C4A92B167EA8FAD081A9F07FD525D52E6DA83F5FBA5386AE9B528,NA,NA,ARA,0.35479,NA,2008,2000/01


In [211]:
home_table_merge <- home_table_merge |> 
    filter(Year %in% c('2013','2014','2015','2016','2017','2018'))

In [212]:
home_table_merge |> distinct(Year) |> pull(Year)

[1] "2017" "2013" "2018" "2014" "2015" "2016"

In [213]:
# where cohort 2001/2 drop 2013 year records
home_table_merge <- home_table_merge |> 
    filter(!(cohort == '2001/02' & Year == '2013'))

In [214]:
# where cohort 2000/1 drop 2018 year records
home_table_merge <- home_table_merge |> 
    filter(!(cohort == '2000/01' & Year == '2018'))

In [215]:
home_table_merge |>
    nrow()

[1] 191543

In [216]:
home_table_merge |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
16936


In [217]:
# drop rows per year with NAs
home_table_merge <- home_table_merge |>
    arrange(desc(Year)) |>
    group_by(person_id, Year) |>
    summarise(across(everything(), ~ ifelse(all(is.na(.)), NA, na.omit(.)[1])), .groups = "drop")

In [218]:
home_table_merge |>
    nrow()

[1] 82262

In [219]:
home_table_merge |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
16936


In [220]:
home_table_merge |>
    filter(person_id == '0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD')

person_id,Year,DistCurrSch,DistNearSch,Language,IDACIScore_Home,LSOA11_Home,cohort
<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2013,0.66,0.66,ENG,0.32444,E01010635,2000/01
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2014,0.66,0.66,ENG,0.32444,E01010635,2000/01
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2015,0.66,0.66,ENG,0.32444,E01010635,2000/01
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2016,0.66,0.66,ENG,NA,E01010635,2000/01
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017,0.66,0.66,ENG,0.41000,E01010635,2000/01


In [221]:
home_table_merge |>
    filter(person_id == '0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D')

person_id,Year,DistCurrSch,DistNearSch,Language,IDACIScore_Home,LSOA11_Home,cohort
<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2014,1.59,0.81,ENG,0.20557,E01002616,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2015,1.59,0.48,ENG,0.20557,E01002616,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2016,1.59,0.48,ENG,NA,E01002616,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2017,7.01,0.00,ENG,0.12500,E01002563,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018,7.01,0.00,ENG,NA,E01002563,2001/02


In [222]:
# group by all cols except year, and keep first row of duplicates
home_table_merge <- home_table_merge |>
    arrange(desc(Year)) |>
    group_by(across(-Year)) |>
    filter(row_number() == 1) |>
    ungroup()

In [223]:
home_table_merge |>
    nrow()

[1] 61118

In [224]:
home_table_merge |>
    filter(person_id == '0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD')

person_id,Year,DistCurrSch,DistNearSch,Language,IDACIScore_Home,LSOA11_Home,cohort
<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017,0.66,0.66,ENG,0.41000,E01010635,2000/01
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2016,0.66,0.66,ENG,NA,E01010635,2000/01
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2015,0.66,0.66,ENG,0.32444,E01010635,2000/01


In [225]:
home_table_merge |>
    filter(person_id == '0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D')

person_id,Year,DistCurrSch,DistNearSch,Language,IDACIScore_Home,LSOA11_Home,cohort
<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018,7.01,0.00,ENG,NA,E01002563,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2017,7.01,0.00,ENG,0.12500,E01002563,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2016,1.59,0.48,ENG,NA,E01002616,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2015,1.59,0.48,ENG,0.20557,E01002616,2001/02
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2014,1.59,0.81,ENG,0.20557,E01002616,2001/02


In [226]:
# repeat merge rows where there are NA values keep first non NA (latest date prioritised)
home_table_merge <- home_table_merge |>
    arrange(desc(Year)) |>
    group_by(person_id) |>
    summarise(across(everything(), ~ ifelse(all(is.na(.)), NA, na.omit(.)[1])), .groups = "drop")

In [227]:
head(home_table_merge)

person_id,Year,DistCurrSch,DistNearSch,Language,IDACIScore_Home,LSOA11_Home,cohort
<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017,0.66,0.66,ENG,0.410,E01010635,2000/01
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2017,3.50,0.62,ENG,0.397,E01010749,2000/01
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018,7.01,0.00,ENG,0.125,E01002563,2001/02
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2018,0.57,0.57,PNJ,0.405,E01010724,2001/02
00149D26484ED3BE2DEF154B8DB53A36B1A9FF0839C447A2CE19B0750225BA66,2017,1.69,0.44,PNJ,0.521,E01010610,2000/01
0014DF7EADF7C22A0FD2F6862B6F06778B199F1F2D61242125DA8EC539E6275D,2017,1.12,0.62,ENG,0.234,E01010815,2000/01


In [228]:
home_table_merge |>
    filter(person_id == '0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD')

person_id,Year,DistCurrSch,DistNearSch,Language,IDACIScore_Home,LSOA11_Home,cohort
<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017,0.66,0.66,ENG,0.41,E01010635,2000/01


In [229]:
home_table_merge |>
    filter(person_id == '0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D')

person_id,Year,DistCurrSch,DistNearSch,Language,IDACIScore_Home,LSOA11_Home,cohort
<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018,7.01,0,ENG,0.125,E01002563,2001/02


In [230]:
home_table_merge |> nrow()

[1] 16936

In [231]:
home_table_merge |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
16936


#### EAL 

In [232]:
home_table_merge <- home_table_merge |>
    mutate(
    EAL = case_when(
      Language == "ENG" ~ 0,
      is.na(Language) ~ NA,
      TRUE ~ 1 
    )
  )  

In [233]:
home_table_merge <- home_table_merge |>
    select(-c(Language, Year))

In [234]:
head(home_table_merge)

person_id,DistCurrSch,DistNearSch,IDACIScore_Home,LSOA11_Home,cohort,EAL
<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,0.66,0.66,0.410,E01010635,2000/01,0
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,3.50,0.62,0.397,E01010749,2000/01,0
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,7.01,0.00,0.125,E01002563,2001/02,0
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,0.57,0.57,0.405,E01010724,2001/02,1
00149D26484ED3BE2DEF154B8DB53A36B1A9FF0839C447A2CE19B0750225BA66,1.69,0.44,0.521,E01010610,2000/01,1
0014DF7EADF7C22A0FD2F6862B6F06778B199F1F2D61242125DA8EC539E6275D,1.12,0.62,0.234,E01010815,2000/01,0


## merge person and home

In [235]:
home_table_merge <- home_table_merge |>
    select(-cohort)

In [236]:
person_home <- person_filtered |> 
    left_join(home_table_merge, by = join_by(person_id)) 

In [237]:
head(person_home)

,person_id,birth_date,gender,CombinedEthnicity,cohort,DistCurrSch,DistNearSch,IDACIScore_Home,LSOA11_Home,EAL
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>
1,D27CE553B8DA52E66BCBA43420F6F53DAEBF9FE2864361849339028B6B002695,2000-09-15,F,South Asian,2000/01,0.90,0.85,0.245,E01010662,1
2,9F33F6D292805042AF92186788B71C7302C7E976F6C148C34FDE62B157C03928,2000-09-15,F,South Asian,2000/01,3.87,0.37,0.236,E01010743,0
3,C38AD0B73560547A95C529E42A9CF6E1E06282163BB419DC2C1451422CD20CE0,2000-09-15,F,South Asian,2000/01,0.60,0.60,0.401,E01010745,1
4,8C9979BE485657CDE5AC92361D6E454DC61C1396BCC38F77933217649F802EFF,2000-09-15,F,White British,2000/01,2.72,2.72,0.049,E01031883,0
5,00ED2E8CBC0AE48B370DBB70FB9440A00BD65113392DE33A5E14380D5C306075,2000-09-15,F,White British,2000/01,1.48,1.48,0.080,E01010795,0
6,F9AA50A652E66CCDB96FE50B68A1035B2612296C86D2B839A97D5B3533FEA597,2000-09-15,F,White British,2000/01,0.25,0.25,0.171,E01017898,0


In [238]:
# save as csv
write.csv(person_home, "data/person_home.csv", row.names = FALSE)

# Disability

In [236]:
student_data = "CB_2489.cb_Student"

student_table <- tbl(con, student_data) |> 
    select(person_id, Gender, Disability) |>
    filter(!is.na(person_id)) |>
    filter(!is.na(Disability))

In [237]:
student_table

# Source:   SQL [?? x 3]
# Database: BigQueryConnection
   person_id                                                   Gender Disability
   <chr>                                                       <chr>  <chr>     
 1 1BB207623557A2E675505C4CA5237B7AB9EB8EA3B3D17E70CD93A68926… F      NCOL      
 2 26E90FE4263275E646264F68FA34C104CD1AEE6BF45CF0D7F256E117A4… F      NCOL      
 3 10BC24A12C475E751290F1DD6CE0129AE5627F2676F2E7041EC4D69026… F      NCOL      
 4 EF5BDEACB889527A7BC6F6F29DC7A3CB88F77A7AF5A12E8B6DE9F88F73… F      NCOL      
 5 F7FDCCAFC13BA5A4F2FC7E407D12F14C1ACE57A0DB38874412D284C4C1… M      NCOL      
 6 D2630283F9C4DFDB64DA4509C55BA1ACC3E62B0D50FFD12CB01721F495… M      NCOL      
 7 943242326C63FCA6AAF153271971640F2CDC932543D81CB0A3E3022D93… M      NCOL      
 8 EA6F60F9824E3D26D76A78F9EC54279A433AE4390DB2953CCAB34F0300… M      NCOL      
 9 24BCE3FCC408EFDB17B492DE055E5E7E7DB013199B3768B60257B4375D… M      NCOL      
10 F27B849A12EB97ECE880026DB0E694F30594669BA0870B3DA7

In [238]:
student_df <- collect(student_table)

In [239]:
# filter person to only matching IDs
student_filtered <- student_df |>
    filter(person_id %in% person_ids$person_id)

In [242]:
head(student_filtered)

person_id,Gender,Disability
<chr>,<chr>,<chr>
1BB207623557A2E675505C4CA5237B7AB9EB8EA3B3D17E70CD93A68926983A83,F,NCOL
26E90FE4263275E646264F68FA34C104CD1AEE6BF45CF0D7F256E117A4575042,F,NCOL
10BC24A12C475E751290F1DD6CE0129AE5627F2676F2E7041EC4D690264BB58D,F,NCOL
EF5BDEACB889527A7BC6F6F29DC7A3CB88F77A7AF5A12E8B6DE9F88F7397AC87,F,NCOL
D2630283F9C4DFDB64DA4509C55BA1ACC3E62B0D50FFD12CB01721F495611C5E,M,NCOL
EA6F60F9824E3D26D76A78F9EC54279A433AE4390DB2953CCAB34F03007267F2,M,NCOL


In [244]:
student_filtered |> 
  distinct(Disability) |> 
  pull(Disability)

[1] "NCOL" "BEH"  "NONE" "LD"   "MOB"  "COMM" "HEAR" "AUT"  "CON"  "OTH" 
[11] "VIS"  "HAND" "MED"  "EAT"  "PC"

In [245]:
student_table <- student_filtered |> 
  mutate(
    SEN = case_when(
      Disability %in% c('NCOL', "", NA) ~ NA_character_,
      TRUE ~ Disability  # Handle any other unexpected cases
    )
  ) 

In [246]:
head(student_table)

person_id,Gender,Disability,SEN
<chr>,<chr>,<chr>,<chr>
1BB207623557A2E675505C4CA5237B7AB9EB8EA3B3D17E70CD93A68926983A83,F,NCOL,NA
26E90FE4263275E646264F68FA34C104CD1AEE6BF45CF0D7F256E117A4575042,F,NCOL,NA
10BC24A12C475E751290F1DD6CE0129AE5627F2676F2E7041EC4D690264BB58D,F,NCOL,NA
EF5BDEACB889527A7BC6F6F29DC7A3CB88F77A7AF5A12E8B6DE9F88F7397AC87,F,NCOL,NA
D2630283F9C4DFDB64DA4509C55BA1ACC3E62B0D50FFD12CB01721F495611C5E,M,NCOL,NA
EA6F60F9824E3D26D76A78F9EC54279A433AE4390DB2953CCAB34F03007267F2,M,NCOL,NA


In [356]:
student_table |>
    group_by(SEN) |>
    tally() |>
    arrange(desc(n))

SEN,n
<chr>,<int>
NA,15788
NONE,300
LD,64
BEH,21
OTH,15
AUT,13
COMM,10
HEAR,7
EAT,4


## SEN 

In [3]:
sen_data = "CB_2489.cb_StudentEntitlementFromCensus"

sen_table <- tbl(con, sen_data) |> 
    select(person_id, SourceTable, EVERFSM_ALL, SENStage, SENprovision, SENprovisionMajor, PrimarySENtype) |>
    filter(!is.na(person_id))

In [4]:
head(sen_table)

# Source:   SQL [6 x 7]
# Database: BigQueryConnection
  person_id      SourceTable EVERFSM_ALL SENStage SENprovision SENprovisionMajor
  <chr>          <chr>             <dbl> <chr>    <chr>        <chr>            
1 B1D77C27358C0… PLASC_2002           NA 0        NA           NA               
2 B0F6576C4FD11… PLASC_2002           NA 2        NA           NA               
3 743AE9CEC04DB… PLASC_2002           NA 0        NA           NA               
4 F8997374AD18D… PLASC_2002           NA 2        NA           NA               
5 F55EE5CB23250… PLASC_2002           NA 0        NA           NA               
6 85F82007BB545… PLASC_2002           NA 0        NA           NA               
# ℹ 1 more variable: PrimarySENtype <chr>

In [5]:
sen_table |> 
  distinct(SourceTable) |> 
  pull(SourceTable)

[1] "PLASC_2002"          "PLASC_2005"          "Autumn_Census_2007" 
 [4] "Autumn_Census_2009"  "Autumn_Census_2010"  "Autumn_Census_2012" 
 [7] "Autumn_Census_2013"  "Autumn_Census_2014"  "Spring_Census_2006" 
[10] "Spring_Census_2009"  "Spring_Census_2015"  "Summer_Census_2008" 
[13] "Summer_Census_2010"  "Summer_Census_2012"  "Summer_Census_2013" 
[16] "Summer_Census_2015"  "Autumn_Census_2018"  "Autumn_Census_2019" 
[19] "Autumn_Census_2020"  "Autumn_Census_2021"  "Spring_Census_2016" 
[22] "Summer_Census_2016"  "Summer_Census_20017" "PLASC_2003"         
[25] "PLASC_2004"          "Autumn_Census_2008"  "Autumn_Census_2011" 
[28] "Autumn_Census_2015"  "SUMMER_Census_2006"  "Spring_Census_2007" 
[31] "Spring_Census_2008"  "Spring_Census_2010"  "Spring_Census_2011" 
[34] "Spring_Census_2012"  "Spring_Census_2013"  "Spring_Census_2014" 
[37] "Summer_Census_2007"  "Summer_Census_2009"  "Summer_Census_2011" 
[40] "Summer_Census_2014"  "Autumn_Census_2016"  "Autumn_Census_2017" 
[43] "Autumn_Census_2022"  "Spring_Census_2019"  "Spring_Census_2020" 
[46] "Spring_Census_2021"  "Spring_Census_2022"  "Summer_Census_2018" 
[49] "Summer_Census_2019"  "Summer_Census_2021"  "Summer_Census_2022"

Pupil Level Annual School Census (PLASC)

In [6]:
sen_df <- collect(sen_table)

In [241]:
# filter to person_ids
sen_filtered <- sen_df |>
    filter(person_id %in% person_ids$person_id)

In [242]:
# edit label for Summer_Census_20017
sen_filtered <- sen_filtered |>
    mutate(SourceTable = case_when(
        SourceTable == 'Summer_Census_20017' ~ 'Summer_Census_2017',
        TRUE ~ SourceTable)
           )

In [243]:
# extract year from source table
sen_filtered$Year <- substr(sen_filtered$SourceTable, nchar(sen_filtered$SourceTable) - 3, nchar(sen_filtered$SourceTable))

In [244]:
sen_filtered |>
    arrange(Year) |>
    filter(person_id == 'D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E')

person_id,SourceTable,EVERFSM_ALL,SENStage,SENprovision,SENprovisionMajor,PrimarySENtype,Year
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,Summer_Census_2007,NA,NA,N,NA,NA,2007
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,Spring_Census_2007,NA,NA,N,NA,NA,2007
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,Autumn_Census_2008,NA,NA,N,NA,NA,2008
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,Summer_Census_2008,NA,NA,A,NA,NA,2008
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,Spring_Census_2008,NA,NA,A,NA,NA,2008
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,Autumn_Census_2009,NA,NA,A,NA,NA,2009
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,Spring_Census_2009,NA,NA,A,2_SNS,NA,2009
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,Summer_Census_2009,NA,NA,A,2_SNS,NA,2009
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,Spring_Census_2010,0,NA,P,2_SNS,SPLD,2010


In [245]:
head(sen_filtered)

person_id,SourceTable,EVERFSM_ALL,SENStage,SENprovision,SENprovisionMajor,PrimarySENtype,Year
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>
78A31FE77B8702EC379E452B50B3D3C195ECE9EC5E11E564D61F2C58BC1097B2,PLASC_2005,NA,NA,A,NA,NA,2005
E964E0F5400562D02CB0792EC9B83867E6D88DB190C65E6C57B6800F65925C32,Autumn_Census_2008,NA,NA,A,NA,NA,2008
C62F1A01AE50FD58284D8BDCB7A55807DEE2EC6AC977BBAA7E42C9A14EE8C984,Autumn_Census_2008,NA,NA,A,NA,NA,2008
6B939B60E0C7582155BCC0EE14E628375B483FD73456FBC9E154E5BF0E0F5BFB,Autumn_Census_2008,NA,NA,A,NA,NA,2008
03DBAB2F9652CA5D031AE4E9DFDF1E1B449B93416D87DCF6F3FADB708C17A4E6,Autumn_Census_2008,NA,NA,A,NA,NA,2008
EF09AEE797A50A1A115E37691E61DE9B912780D0F0D7C6E8591347821B243374,Autumn_Census_2008,NA,NA,A,NA,NA,2008


In [246]:
sen_filtered |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17224


#### extract only FSM col

In [247]:
fsm <- sen_filtered |>
    select(person_id, Year, EVERFSM_ALL)

In [248]:
fsm_filtered <- fsm |>
    filter(!is.na(EVERFSM_ALL))

In [249]:
fsm_filtered |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17009


In [250]:
# keep latest year only
fsm_filtered <- fsm_filtered |>
      arrange(person_id, desc(Year)) |>
      distinct(person_id, .keep_all = TRUE)

In [251]:
fsm_filtered|>
    nrow()

[1] 17009

In [252]:
fsm_filtered |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17009


In [253]:
fsm_filtered |>
    group_by(EVERFSM_ALL) |>
    tally()

EVERFSM_ALL,n
<dbl>,<int>
0,10142
1,6867


#### merge in FSM col to main data

In [254]:
fsm_filtered <- fsm_filtered |>
    select(person_id, EVERFSM_ALL) |>
    rename(ever_fsm = EVERFSM_ALL)

In [255]:
person_home_fsm <- person_home |> 
    left_join(fsm_filtered, by = join_by(person_id)) 

In [256]:
head(person_home_fsm)

,person_id,birth_date,gender,CombinedEthnicity,cohort,DistCurrSch,DistNearSch,IDACIScore_Home,LSOA11_Home,EAL,ever_fsm
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>
1,D27CE553B8DA52E66BCBA43420F6F53DAEBF9FE2864361849339028B6B002695,2000-09-15,F,South Asian,2000/01,0.90,0.85,0.245,E01010662,1,0
2,9F33F6D292805042AF92186788B71C7302C7E976F6C148C34FDE62B157C03928,2000-09-15,F,South Asian,2000/01,3.87,0.37,0.236,E01010743,0,0
3,C38AD0B73560547A95C529E42A9CF6E1E06282163BB419DC2C1451422CD20CE0,2000-09-15,F,South Asian,2000/01,0.60,0.60,0.401,E01010745,1,1
4,8C9979BE485657CDE5AC92361D6E454DC61C1396BCC38F77933217649F802EFF,2000-09-15,F,White British,2000/01,2.72,2.72,0.049,E01031883,0,0
5,00ED2E8CBC0AE48B370DBB70FB9440A00BD65113392DE33A5E14380D5C306075,2000-09-15,F,White British,2000/01,1.48,1.48,0.080,E01010795,0,0
6,F9AA50A652E66CCDB96FE50B68A1035B2612296C86D2B839A97D5B3533FEA597,2000-09-15,F,White British,2000/01,0.25,0.25,0.171,E01017898,0,0


#### SEN cols

In [15]:
sen_filtered <- sen_filtered |>
    select(-c(EVERFSM_ALL)) |>
    arrange(person_id, desc(Year))

In [16]:
head(sen_filtered)

person_id,SourceTable,SENStage,SENprovision,SENprovisionMajor,PrimarySENtype,Year
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Autumn_Census_2017,NA,K,2_SNS,NA,2017
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Summer_Census_2017,NA,K,2_SNS,NA,2017
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Spring_Census_2016,NA,K,2_SNS,ASD,2016
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Summer_Census_2016,NA,K,2_SNS,NA,2016
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Autumn_Census_2016,NA,N,1_NON,NA,2016
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,Autumn_Census_2015,NA,N,1_NON,NA,2015


In [17]:
sen_filtered |>
    distinct(Year) |> 
    pull(Year)

[1] "2017" "2016" "2015" "2014" "2013" "2012" "2011" "2010" "2009" "2008"
[11] "2007" "2006" "2005" "2018" "2019" "2004" "2021" "2020" "2022" "2003"

In [18]:
sen_filtered_year <- sen_filtered |>
    filter(Year %in% c('2020','2019','2018','2017','2016','2015','2014','2013','2012','2011','2010','2009','2008','2007','2006','2005'))

In [19]:
sen_filtered_year |>
    nrow()

[1] 568825

In [20]:
sen_filtered_year |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17224


In [103]:
sen_2020 <- sen_filtered_year |>
    filter(Year == '2020') |>
    filter(SENprovisionMajor != '1_NON')

In [108]:
all_2020 <- sen_filtered_year |>
    filter(Year == '2020') 

In [106]:
no2020 <- sen_filtered_year |>
    filter(Year != '2020')

In [107]:
# check for any ids in 2020 but not in previous years - if 0 then can drop 2020 (post graduation)
sen_2020_only <- anti_join(sen_2020, no2020, by = "person_id")
sen_2020_only

person_id,SENStage,SENprovision,SENprovisionMajor,PrimarySENtype,Year
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>


In [109]:
all_2020_only <- anti_join(all_2020, no2020, by = "person_id")
all_2020_only

person_id,SENStage,SENprovision,SENprovisionMajor,PrimarySENtype,Year
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
536620403E72C1652C40AB937B9A859788C9E4514D726C6CE941706924977A13,NA,N,1_NON,NA,2020


In [110]:
sen_filtered_year <- sen_filtered_year |>
    filter(Year %in% c('2019','2018','2017','2016','2015','2014','2013','2012','2011','2010','2009','2008','2007','2006','2005'))

In [111]:
# only keep the 2020 row for one person with data only in 2020 which is no sen anyway (rather than having unknown for this id)
sen_filtered_year <- rbind(sen_filtered_year,all_2020_only)

In [112]:
sen_filtered_year |>
    nrow()

[1] 261248

In [113]:
sen_filtered_year |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17224


In [489]:
#missing_ids <- anti_join(person_ids, sen_filtered_year, by = "person_id") %>%
#  select(person_id) %>%
#  distinct()

In [490]:
#sen_filtered |>
#    filter(person_id %in% missing_ids$person_id) |>
#    distinct(Year) |>
#    pull(Year)

In [491]:
#sen_filtered |>
#    filter(person_id %in% missing_ids$person_id) |>
#    filter(Year =='2005') |>
#    summarise(count = n_distinct(person_id))

In [492]:
#sen_filtered |>
#    filter(person_id %in% missing_ids$person_id) |>
#    filter(Year =='2005')

In [114]:
sen_filtered |>
    filter(person_id =='D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848')

person_id,SENStage,SENprovision,SENprovisionMajor,PrimarySENtype,Year
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,2_SNS,NA,2011
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,2_SNS,SLCN,2011
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,N,1_NON,NA,2010
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,2_SNS,SPLD,2010
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,2_SNS,NA,2010
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,NA,NA,2009
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,N,1_NON,NA,2009
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,NA,MLD,2008
D2C1319A064B0552A68F440AE60726DB822B2FFFF28F22B134C3EE9937A8A848,NA,P,NA,NA,2008


In [119]:
# leave year arrange as asc so that first instance of new status remains 
sen_cut <- sen_filtered_year |>
    arrange(Year) |>
    group_by(person_id, Year) |>
    summarise(across(everything(), ~ ifelse(all(is.na(.)), NA, na.omit(.)[1])), .groups = "drop")

In [120]:
sen_cut |>
    nrow()

[1] 217384

In [121]:
sen_cut |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17224


In [122]:
sen_cut |>
    filter(person_id == '0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD')

person_id,Year,SENStage,SENprovision,SENprovisionMajor,PrimarySENtype
<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2005,NA,N,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2006,NA,N,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2007,NA,N,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2008,NA,N,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2009,NA,N,1_NON,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2010,NA,N,1_NON,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2011,NA,N,1_NON,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2012,NA,N,1_NON,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2013,NA,N,1_NON,NA


In [123]:
sen_cut |>
    filter(person_id == '4C4D6048A5783B8393F449A793B295C3A592D7FBCE4D44E2BFD93D25F6A96E69')

person_id,Year,SENStage,SENprovision,SENprovisionMajor,PrimarySENtype
<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
4C4D6048A5783B8393F449A793B295C3A592D7FBCE4D44E2BFD93D25F6A96E69,2006,NA,N,NA,NA
4C4D6048A5783B8393F449A793B295C3A592D7FBCE4D44E2BFD93D25F6A96E69,2007,NA,A,NA,NA
4C4D6048A5783B8393F449A793B295C3A592D7FBCE4D44E2BFD93D25F6A96E69,2008,NA,A,NA,NA
4C4D6048A5783B8393F449A793B295C3A592D7FBCE4D44E2BFD93D25F6A96E69,2009,NA,A,2_SNS,NA
4C4D6048A5783B8393F449A793B295C3A592D7FBCE4D44E2BFD93D25F6A96E69,2010,NA,A,2_SNS,NA
4C4D6048A5783B8393F449A793B295C3A592D7FBCE4D44E2BFD93D25F6A96E69,2011,NA,A,2_SNS,NA
4C4D6048A5783B8393F449A793B295C3A592D7FBCE4D44E2BFD93D25F6A96E69,2012,NA,A,2_SNS,NA
4C4D6048A5783B8393F449A793B295C3A592D7FBCE4D44E2BFD93D25F6A96E69,2013,NA,A,2_SNS,NA
4C4D6048A5783B8393F449A793B295C3A592D7FBCE4D44E2BFD93D25F6A96E69,2014,NA,A,2_SNS,NA


In [124]:
head(sen_cut)

person_id,Year,SENStage,SENprovision,SENprovisionMajor,PrimarySENtype
<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2005,NA,N,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2006,NA,N,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2007,NA,N,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2008,NA,N,NA,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2009,NA,N,1_NON,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2010,NA,N,1_NON,NA


#### clean SEN cols

In [125]:
sen_cut |> 
    nrow()

[1] 217384

In [126]:
sen_cut |>
    group_by(SENStage) |>
    tally()

SENStage,n
<lgl>,<int>
NA,217384


In [127]:
sen_clean <- sen_cut |>
    select(-SENStage)

In [128]:
sen_clean |>
    group_by(SENprovision) |>
    tally() |>
    arrange(desc(n))

SENprovision,n
<chr>,<int>
N,175189
A,20926
P,9563
K,7049
S,3627
E,1028
e,2


In [129]:
sen_clean <- sen_clean |>
    mutate(SENprovisionNEW = case_when(
        SENprovision %in% c('N') ~ 'No SEN',
        SENprovision %in% c('A','P','K') ~ 'SEN support',
        SENprovision %in% c('S','E') ~ 'EHC plan',
        TRUE ~ NA
            )
         )

In [130]:
head(sen_clean)

person_id,Year,SENprovision,SENprovisionMajor,PrimarySENtype,SENprovisionNEW
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2005,N,NA,NA,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2006,N,NA,NA,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2007,N,NA,NA,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2008,N,NA,NA,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2009,N,1_NON,NA,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2010,N,1_NON,NA,No SEN


In [131]:
sen_clean |>
    group_by(SENprovisionMajor) |>
    tally() 

SENprovisionMajor,n
<chr>,<int>
1_NON,129857
2_SNS,31503
3_SS,4184
NA,51840


In [132]:
sen_clean <- sen_clean |>
    select(-SENprovision) |>
    mutate(SENmajorNEW = case_when(
        SENprovisionMajor == '1_NON' ~ 'No SEN',
        SENprovisionMajor == '2_SNS' ~ 'SEN support',
        SENprovisionMajor == '3_SS' ~ 'EHC plan',
        TRUE ~ NA
            )
         )

In [133]:
head(sen_clean)

person_id,Year,SENprovisionMajor,PrimarySENtype,SENprovisionNEW,SENmajorNEW
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2005,NA,NA,No SEN,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2006,NA,NA,No SEN,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2007,NA,NA,No SEN,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2008,NA,NA,No SEN,NA
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2009,1_NON,NA,No SEN,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2010,1_NON,NA,No SEN,No SEN


In [134]:
# check where two new cols dont match
sen_clean |>
    filter(SENprovisionNEW != SENmajorNEW & !is.na(SENprovisionNEW) & !is.na(SENmajorNEW))

person_id,Year,SENprovisionMajor,PrimarySENtype,SENprovisionNEW,SENmajorNEW
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0054192801954D860BB235F844AD1CB8A99956B2CF3D4E20F83C650DD6279B58,2009,1_NON,NA,SEN support,No SEN
01B893444938704BE6DF40B1FDF71CE4741DF8F2A102BA6068B1401827A8BA1E,2009,2_SNS,NA,No SEN,SEN support
01DD211111ECE9C51A992B85C0E53F8F3A604974F18AD8B5C0C2A2071C05D8DC,2009,2_SNS,SLD,No SEN,SEN support
02186E060FD3D7B6FE18E6592CB04AF4CD2ADAA753D0CE3CDA81D2D8479CABEA,2009,1_NON,NA,SEN support,No SEN
024B506650AA53563BCC08BBC93D7746728699078E4FEAFA951991731066A93B,2009,2_SNS,NA,No SEN,SEN support
035392BAC7F954291C7B2B244E0225041403D4603261ABFDAAEC64C4A2C6A6AD,2009,2_SNS,NA,No SEN,SEN support
062C48DE2C8CCADC19321E70514E5A072661A4E07A7EC55FC2427710CA555127,2009,2_SNS,NA,No SEN,SEN support
06855F3F21AD13B045330156289C42DD01E690C2ECD6BF45A330BF4FDBE166EE,2009,1_NON,NA,SEN support,No SEN
068E6217CC38D05AEEAFBC7827F078C4242EDC81CA0A27D4444E320868269411,2009,1_NON,NA,SEN support,No SEN


In [135]:
# merge values across two new cols
# where one of the new cols is no sen and the other is sen, keep sen
sen_clean <- sen_clean |>
    mutate(SENlevel = case_when(
        SENprovisionNEW == SENmajorNEW ~ SENprovisionNEW,  # both are the same
        is.na(SENprovisionNEW) & !is.na(SENmajorNEW) ~ SENmajorNEW,  # drop NA
        !is.na(SENprovisionNEW) & is.na(SENmajorNEW) ~ SENprovisionNEW,  # drop NA
        SENprovisionNEW == "No SEN" & SENmajorNEW == "SEN support" ~ SENmajorNEW,  # SEN support over No SEN
        SENprovisionNEW == "SEN support" & SENmajorNEW == "No SEN" ~ SENprovisionNEW,  # SEN support over No SEN
        SENprovisionNEW == "SEN support" & SENmajorNEW == "EHC plan" ~ SENmajorNEW,  # EHC plan over SEN support 
        SENprovisionNEW == "EHC plan" & SENmajorNEW == "SEN support" ~ SENprovisionNEW,  # EHC plan over SEN support 
        SENprovisionNEW == "No SEN" & SENmajorNEW == "EHC plan" ~ SENmajorNEW,  # EHC plan over SEN support 
        TRUE ~ NA
  ))

In [136]:
# check where two new cols dont match incase of EHCP col not matching
sen_clean |>
    filter(is.na(SENlevel))

person_id,Year,SENprovisionMajor,PrimarySENtype,SENprovisionNEW,SENmajorNEW,SENlevel
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>


In [137]:
sen_clean <- sen_clean |>
    select(-SENprovisionMajor)

In [138]:
sen_clean |>
    filter(person_id == 'A26C103463F0FCEC6212A7BCE763E627E221E9F0ABDF45246C92E7BC89F9F23E')

person_id,Year,PrimarySENtype,SENprovisionNEW,SENmajorNEW,SENlevel
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
A26C103463F0FCEC6212A7BCE763E627E221E9F0ABDF45246C92E7BC89F9F23E,2005,MLD,SEN support,NA,SEN support
A26C103463F0FCEC6212A7BCE763E627E221E9F0ABDF45246C92E7BC89F9F23E,2006,NA,No SEN,NA,No SEN
A26C103463F0FCEC6212A7BCE763E627E221E9F0ABDF45246C92E7BC89F9F23E,2007,NA,SEN support,NA,SEN support
A26C103463F0FCEC6212A7BCE763E627E221E9F0ABDF45246C92E7BC89F9F23E,2008,SLCN,SEN support,NA,SEN support
A26C103463F0FCEC6212A7BCE763E627E221E9F0ABDF45246C92E7BC89F9F23E,2009,SLCN,EHC plan,EHC plan,EHC plan
A26C103463F0FCEC6212A7BCE763E627E221E9F0ABDF45246C92E7BC89F9F23E,2010,SLCN,EHC plan,EHC plan,EHC plan
A26C103463F0FCEC6212A7BCE763E627E221E9F0ABDF45246C92E7BC89F9F23E,2011,SLCN,EHC plan,EHC plan,EHC plan
A26C103463F0FCEC6212A7BCE763E627E221E9F0ABDF45246C92E7BC89F9F23E,2012,SLCN,EHC plan,EHC plan,EHC plan
A26C103463F0FCEC6212A7BCE763E627E221E9F0ABDF45246C92E7BC89F9F23E,2013,MLD,EHC plan,EHC plan,EHC plan


In [139]:
sen_clean |>
    filter(person_id == '0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7')

person_id,Year,PrimarySENtype,SENprovisionNEW,SENmajorNEW,SENlevel
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2006,NA,No SEN,NA,No SEN
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2007,NA,No SEN,NA,No SEN
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2008,NA,No SEN,NA,No SEN
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2009,NA,No SEN,No SEN,No SEN
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2010,NA,No SEN,No SEN,No SEN
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2011,NA,No SEN,No SEN,No SEN
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2012,NA,No SEN,No SEN,No SEN
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2013,NA,No SEN,No SEN,No SEN
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2014,MLD,SEN support,SEN support,SEN support


In [140]:
sen_clean <- sen_clean |>
    select(-c(SENprovisionNEW,SENmajorNEW))

In [141]:
sen_clean

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2005,NA,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2006,NA,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2007,NA,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2008,NA,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2009,NA,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2010,NA,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2011,NA,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2012,NA,No SEN
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2013,NA,No SEN


In [147]:
# copy previous value where sen level is the same but type is NA
sen_clean <- sen_clean |>
    arrange(person_id, Year) |>  
    group_by(person_id) |>
    mutate(PrimarySENtype = case_when(
          # if NA and SENlevel is same, and prev not NA take prev value
          is.na(PrimarySENtype) & lag(SENlevel) == SENlevel & !is.na(lag(PrimarySENtype)) ~ lag(PrimarySENtype),  
          TRUE ~ PrimarySENtype  # Keep original value otherwise
        )) |>
    #fill(PrimarySENtype, .direction = "down") |>
    ungroup()

In [149]:
# repeat
sen_clean <- sen_clean |>
    arrange(person_id, Year) |>  
    group_by(person_id) |>
    mutate(PrimarySENtype = case_when(
          # if NA and SENlevel is same, and prev not NA take prev value
          is.na(PrimarySENtype) & lag(SENlevel) == SENlevel & !is.na(lag(PrimarySENtype)) ~ lag(PrimarySENtype),  
          TRUE ~ PrimarySENtype  # Keep original value otherwise
        )) |>
    #fill(PrimarySENtype, .direction = "down") |>
    ungroup()

What are the implications of having an SEN but it only being identified / support being put in place when in year 11???

In [143]:
person_ids |>
    filter(person_id == '0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD')

person_id,NCCIS_ACADYR
<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018


In [151]:
sen_clean |>
    filter(SENlevel != 'No SEN')

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2016,ASD,SEN support
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017,ASD,SEN support
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2007,MLD,SEN support
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2008,MLD,SEN support
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2009,MLD,SEN support
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2010,MLD,SEN support
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2011,MLD,SEN support
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2012,MLD,SEN support
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2013,MLD,SEN support


In [150]:
sen_clean <- sen_clean |>
    mutate(PrimarySENtype = case_when(
        PrimarySENtype == 'BESD' ~ 'SEMH',
        TRUE ~ PrimarySENtype)
           )

#### join cohort id

In [152]:
sen_clean_cohort <- sen_clean |>
    left_join(person_ids, by = join_by(person_id)) 

In [153]:
sen_clean_cohort

person_id,Year,PrimarySENtype,SENlevel,NCCIS_ACADYR
<chr>,<chr>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2005,NA,No SEN,2017/2018
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2006,NA,No SEN,2017/2018
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2007,NA,No SEN,2017/2018
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2008,NA,No SEN,2017/2018
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2009,NA,No SEN,2017/2018
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2010,NA,No SEN,2017/2018
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2011,NA,No SEN,2017/2018
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2012,NA,No SEN,2017/2018
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2013,NA,No SEN,2017/2018


In [156]:
sen_clean_cohort |>
    filter(person_id == '0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D')

person_id,Year,PrimarySENtype,SENlevel,NCCIS_ACADYR
<chr>,<int>,<chr>,<chr>,<chr>
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2005,SLCN,SEN support,2018/2019
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2007,NA,No SEN,2018/2019
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2008,NA,No SEN,2018/2019
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2009,MLD,SEN support,2018/2019
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2010,MLD,SEN support,2018/2019
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2011,MLD,SEN support,2018/2019
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2012,MLD,EHC plan,2018/2019
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2013,MLD,EHC plan,2018/2019
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2014,MLD,EHC plan,2018/2019


#### check missing sen type against extras from exclusion tables

In [264]:
sen <- read.csv('data/sen_extras.csv', header = TRUE)

In [265]:
head(sen)

,person_id,PrimarySENtype
,<chr>,<chr>
1,13A5184415FCC22BFF3F2EFFD8D3B69FAB1949AFB346B8A15B449CBED2CDB473,BESD
2,02304796B373267AE442202D65322A36B8844DD043A88734014D6434DD7D1F12,BESD
3,04584E57ADBD062B2BA9DB9076CF692E45F5767837BD87EEE4FED6DB521BB84B,OTH
4,66927CC16F370C097C232D8581FB0E92E89F688DE966B50CD1A128B631F9DB6A,MLD
5,778E21A7061D57C78E5986180FAE5D061F1F8B2B69F1925CDD0FCDC4B424AEF1,BESD
6,7D46116A75B15250162ECE83B71941167D4FFAE82D39C508681D25F69C50BF32,BESD


In [266]:
missing_type <- sen_clean_cohort |>
    filter(SENlevel != 'No SEN' & is.na(PrimarySENtype))

In [270]:
missing_type

person_id,Year,PrimarySENtype,SENlevel,NCCIS_ACADYR
<chr>,<int>,<chr>,<chr>,<chr>
0014DF7EADF7C22A0FD2F6862B6F06778B199F1F2D61242125DA8EC539E6275D,2017,NA,SEN support,2017/2018
0018E26A40A6299391164D4918658B0B1248367C762DCE6D2FDF1AE131A9138C,2008,NA,SEN support,2018/2019
0018E26A40A6299391164D4918658B0B1248367C762DCE6D2FDF1AE131A9138C,2009,NA,SEN support,2018/2019
0018E26A40A6299391164D4918658B0B1248367C762DCE6D2FDF1AE131A9138C,2010,NA,SEN support,2018/2019
0018E26A40A6299391164D4918658B0B1248367C762DCE6D2FDF1AE131A9138C,2011,NA,SEN support,2018/2019
0018E26A40A6299391164D4918658B0B1248367C762DCE6D2FDF1AE131A9138C,2012,NA,SEN support,2018/2019
0018E26A40A6299391164D4918658B0B1248367C762DCE6D2FDF1AE131A9138C,2013,NA,SEN support,2018/2019
0018E26A40A6299391164D4918658B0B1248367C762DCE6D2FDF1AE131A9138C,2014,NA,SEN support,2018/2019
001E539C263734CC355646811C93CF85B0C587D2EBA8B0D79290FDE8A75D6267,2013,NA,SEN support,2017/2018


In [267]:
# Check if all person_ids in person_id are in person
any(sen$person_id %in% missing_type$person_id)

[1] TRUE

In [268]:
sum(sen$person_id %in% missing_type$person_id)

[1] 772

In [269]:
sen <- sen |>
    mutate(PrimarySENtype = case_when(
        PrimarySENtype == 'BESD' ~ 'SEMH',
        TRUE ~ PrimarySENtype)
           )

#### compute 'ever' values

In [155]:
sen_clean_cohort <- sen_clean_cohort |> 
  mutate(Year = as.integer(Year))  # Convert Year to integer

In [171]:
sen_final <- sen_clean_cohort |> 
  group_by(person_id) |> 
  summarize(
    ever_SEN_support = any(SENlevel == "SEN support"),
    first_SEN_support_year = ifelse(ever_SEN_support, min(Year[SENlevel == "SEN support"], na.rm = TRUE), NA),
    SEN_support_need = ifelse(ever_SEN_support, paste(na.omit(unique(PrimarySENtype[SENlevel == "SEN support"])), collapse = ", "), NA),

    ever_EHC_plan = any(SENlevel == "EHC plan"),
    first_EHC_plan_year = ifelse(ever_EHC_plan, min(Year[SENlevel == "EHC plan"], na.rm = TRUE), NA),
    EHC_plan_need = ifelse(ever_EHC_plan, paste(na.omit(unique(PrimarySENtype[SENlevel == "EHC plan"])), collapse = ", "), NA),

    no_SEN = all(SENlevel == "No SEN")
  )

In [172]:
sen_final

person_id,ever_SEN_support,first_SEN_support_year,SEN_support_need,ever_EHC_plan,first_EHC_plan_year,EHC_plan_need,no_SEN
<chr>,<lgl>,<int>,<chr>,<lgl>,<int>,<chr>,<lgl>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,TRUE,2016,ASD,FALSE,NA,NA,FALSE
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,TRUE,2007,MLD,FALSE,NA,NA,FALSE
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,TRUE,2005,"SLCN, MLD",TRUE,2012,MLD,FALSE
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,TRUE,2014,MLD,FALSE,NA,NA,FALSE
00149D26484ED3BE2DEF154B8DB53A36B1A9FF0839C447A2CE19B0750225BA66,FALSE,NA,NA,FALSE,NA,NA,TRUE
0014DF7EADF7C22A0FD2F6862B6F06778B199F1F2D61242125DA8EC539E6275D,TRUE,2017,,FALSE,NA,NA,FALSE
0018E26A40A6299391164D4918658B0B1248367C762DCE6D2FDF1AE131A9138C,TRUE,2008,PD,FALSE,NA,NA,FALSE
0019031CD62586D59268772800516ED50CDF1EF7DE46488542BB031850F6E68A,TRUE,2015,SEMH,FALSE,NA,NA,FALSE
001B7CEB7B4CC529920A0B9F5B83D1AC0019E5DC44B3FBB3CEF665C331C0C3B6,FALSE,NA,NA,FALSE,NA,NA,TRUE


In [173]:
sen_final |> nrow()

[1] 17224

#### ARCHIVE filter to one record pp with year as first record of SEN need

In [58]:
# group by all cols except year, and keep first row of duplicates
sen_merge <- sen_clean |>
    arrange(desc(Year)) |>
    group_by(across(-Year)) |>
    filter(row_number() == 1) |>
    ungroup() |>
    arrange(person_id)

In [59]:
sen_merge |>
    filter(person_id == 'D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E')

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2020,NA,No SEN
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2017,SPLD,SEN support
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2014,NA,SEN support


In [60]:
sen_merge |>
arrange(person_id, desc(Year))

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017,ASD,SEN support
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2015,NA,No SEN
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2017,MLD,SEN support
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2006,NA,No SEN
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018,NA,EHC plan
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2017,MLD,EHC plan
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2011,MLD,SEN support
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2008,NA,No SEN
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2005,SLCN,SEN support


In [575]:
# copy previous value where sen level is NA but type not NA
sen_level_clean <- sen_merge |>
    arrange(person_id, Year) |>  
    group_by(person_id) |>
    mutate(SENlevel = case_when(
          # if NA and prev SENlevel is SEN support, and type not NA 
          SENlevel == 'No SEN' & lag(SENlevel) == 'SEN support' & !is.na(PrimarySENtype) ~ 'SEN support',  
          SENlevel == 'No SEN' & lag(SENlevel) == 'EHC plan' & !is.na(PrimarySENtype) ~ 'EHC plan',
          TRUE ~ SENlevel  # Keep original value otherwise
        )) |>
    ungroup()

In [576]:
sen_level_clean |>
    filter(person_id == 'D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E')

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2007,NA,No SEN
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2008,NA,SEN support
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2010,SPLD,SEN support
D7759C6DE1092426F3CAD1789805C59D6DB27F497B8C8C7038E2E1A4985FC11E,2013,SPLD,SEN support


In [585]:
# repeat but for desc year
sen_level_desc <- sen_level_clean |>
    arrange(person_id, desc(Year)) |>  
    group_by(person_id) |>
    mutate(SENlevel = case_when(
          # if NA and prev SENlevel is SEN support, and type not NA 
          SENlevel == 'No SEN' & lag(SENlevel) == 'SEN support' & !is.na(PrimarySENtype) ~ 'SEN support',  
          SENlevel == 'No SEN' & lag(SENlevel) == 'EHC plan' & !is.na(PrimarySENtype) ~ 'EHC plan',
          TRUE ~ SENlevel  # Keep original value otherwise
        )) |>
    ungroup()

In [591]:
# repeat to catch gaps
sen_level_desc <- sen_level_desc |>
    arrange(person_id, Year) |>  
    group_by(person_id) |>
    mutate(SENlevel = case_when(
          # if NA and prev SENlevel is SEN support, and type not NA 
          SENlevel == 'No SEN' & lag(SENlevel) == 'SEN support' & !is.na(PrimarySENtype) ~ 'SEN support',  
          SENlevel == 'No SEN' & lag(SENlevel) == 'EHC plan' & !is.na(PrimarySENtype) ~ 'EHC plan',
          TRUE ~ SENlevel  # Keep original value otherwise
        )) |>
    ungroup()

In [594]:
sen_level_desc |>
    filter(person_id == 'D4D7BBA8B62A4CEB3450B49A97320FCD1E420F8D489BBA9AF698C2124BC88147')

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
D4D7BBA8B62A4CEB3450B49A97320FCD1E420F8D489BBA9AF698C2124BC88147,2006,NA,No SEN
D4D7BBA8B62A4CEB3450B49A97320FCD1E420F8D489BBA9AF698C2124BC88147,2008,SEMH,SEN support
D4D7BBA8B62A4CEB3450B49A97320FCD1E420F8D489BBA9AF698C2124BC88147,2013,SEMH,SEN support
D4D7BBA8B62A4CEB3450B49A97320FCD1E420F8D489BBA9AF698C2124BC88147,2016,SEMH,SEN support


In [595]:
missing_senlevel <- sen_level_desc |>
    filter(SENlevel == 'No SEN' & !is.na(PrimarySENtype))

In [596]:
missing_senlevel |>
    summarize(n_distinct(person_id))

n_distinct(person_id)
<int>
64


In [598]:
sen_level_desc |>
    filter(person_id %in% missing_senlevel$person_id) |>
    distinct(SENlevel)

SENlevel
<chr>
No SEN


In [599]:
sen_level_desc |>
    filter(person_id %in% missing_senlevel$person_id) |>
    arrange(person_id, Year)

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
05BECC2FA9168CC3CC4A8669F2B6BEF1235BB7A7386DDA71141D0444C1A34D87,2005,OTH,No SEN
0607C1DA04A1A88F14F80F5DB04C9112B950A68544F2F21E241F94F3AC7D6E78,2006,NA,No SEN
0607C1DA04A1A88F14F80F5DB04C9112B950A68544F2F21E241F94F3AC7D6E78,2015,OTH,No SEN
137AAFFB957B50C9C3509763EEE31B4AD73661BB93279CBBCCFB377BDE975C8D,2006,NA,No SEN
137AAFFB957B50C9C3509763EEE31B4AD73661BB93279CBBCCFB377BDE975C8D,2015,SPLD,No SEN
1E2DDA056F48950C6617E5D8209A6CB6E5583CD02031694903C9E11461C1B70B,2006,NA,No SEN
1E2DDA056F48950C6617E5D8209A6CB6E5583CD02031694903C9E11461C1B70B,2015,OTH,No SEN
20E40E9631F550A80DC84266D8EA9118EE12F0C0498AEF95D4ED1DAC9A5124ED,2006,NA,No SEN
20E40E9631F550A80DC84266D8EA9118EE12F0C0498AEF95D4ED1DAC9A5124ED,2015,SPLD,No SEN


In [603]:
sen_final <- sen_level_desc |>
    arrange(desc(Year)) |>            
    group_by(person_id) |>            
    filter(row_number() == 1) |>      
    ungroup()   

In [604]:
sen_final |>
    group_by(PrimarySENtype) |>
    tally() 

PrimarySENtype,n
<chr>,<int>
ASD,230
HI,128
MLD,1540
MSI,3
NSA,54
OTH,204
PD,150
PMLD,27
SEMH,1128


In [606]:
head(sen_final)

person_id,Year,PrimarySENtype,SENlevel
<chr>,<chr>,<chr>,<chr>
0BAA7959F866DC200B0DE7C16B5B16DD63D6D6BA086D46BA9D26025AC7FB1FB6,2020,OTH,SEN support
18A1DBCA368AFAADA2DAC40BD35DE4E1756E5C5905C643A9DFD8590ED65BC980,2020,HI,SEN support
18BDC8758A1C2629E1B3FE7677D649961D106A4E3C5371B14B2EE6FBC12D6162,2020,VI,SEN support
191B614E755130087F1A4C6F61D7CA413FC57CDE42DE350168AF9F21079A5A3D,2020,VI,SEN support
1E8EDACA982683915764BFB3DA811D133E1AD9879F0512C019E4307EFF8E2D13,2020,SEMH,SEN support
248B0C2E0DD811AC20D8D8940381A477C3041C6B299278C2F7C44AB3E3E098A0,2020,OTH,SEN support


In [621]:
sen_final <- sen_final |>
    rename(SENtype = PrimarySENtype) |>
    select(-Year)

In [622]:
sen_final |>
    nrow()

[1] 17224

In [623]:
sen_final |>
    summarise(n_distinct(person_id))

n_distinct(person_id)
<int>
17224


#### merge SEN into main data

In [174]:
# save as csv
write.csv(sen_final, "data/sen_final.csv", row.names = FALSE)

In [257]:
person_home_fsm_sen <- sen_final |> 
    left_join(person_home_fsm, by = join_by(person_id)) 

In [258]:
person_home_fsm_sen

person_id,ever_SEN_support,first_SEN_support_year,SEN_support_need,ever_EHC_plan,first_EHC_plan_year,EHC_plan_need,no_SEN,birth_date,gender,CombinedEthnicity,cohort,DistCurrSch,DistNearSch,IDACIScore_Home,LSOA11_Home,EAL,ever_fsm
<chr>,<lgl>,<int>,<chr>,<lgl>,<int>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,TRUE,2016,ASD,FALSE,NA,NA,FALSE,2001-05-15,M,White British,2000/01,0.66,0.66,0.410,E01010635,0,0
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,TRUE,2007,MLD,FALSE,NA,NA,FALSE,2001-01-15,F,White British,2000/01,3.50,0.62,0.397,E01010749,0,0
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,TRUE,2005,"SLCN, MLD",TRUE,2012,MLD,FALSE,2001-11-15,M,White British,2001/02,7.01,0.00,0.125,E01002563,0,1
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,TRUE,2014,MLD,FALSE,NA,NA,FALSE,2002-05-15,M,Other,2001/02,0.57,0.57,0.405,E01010724,1,0
00149D26484ED3BE2DEF154B8DB53A36B1A9FF0839C447A2CE19B0750225BA66,FALSE,NA,NA,FALSE,NA,NA,TRUE,2000-11-15,M,South Asian,2000/01,1.69,0.44,0.521,E01010610,1,0
0014DF7EADF7C22A0FD2F6862B6F06778B199F1F2D61242125DA8EC539E6275D,TRUE,2017,,FALSE,NA,NA,FALSE,2001-04-15,F,White British,2000/01,1.12,0.62,0.234,E01010815,0,0
0018E26A40A6299391164D4918658B0B1248367C762DCE6D2FDF1AE131A9138C,TRUE,2008,PD,FALSE,NA,NA,FALSE,2002-07-15,M,South Asian,2001/02,0.15,0.15,0.277,E01033693,1,1
0019031CD62586D59268772800516ED50CDF1EF7DE46488542BB031850F6E68A,TRUE,2015,SEMH,FALSE,NA,NA,FALSE,2001-12-15,F,Other,2001/02,0.57,0.57,0.487,E01010663,1,1
001B7CEB7B4CC529920A0B9F5B83D1AC0019E5DC44B3FBB3CEF665C331C0C3B6,FALSE,NA,NA,FALSE,NA,NA,TRUE,2001-05-15,M,White British,2000/01,1.56,1.56,0.034,E01010575,0,0


In [259]:
# save as csv
write.csv(person_home_fsm_sen, "data/person_home_fsm_sen.csv", row.names = FALSE)